In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

In [6]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipeline.pkl"

def build_pipeline(numerical_attribs, categorical_attribs):
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    categorical_pipeline = Pipeline([
        ("Onehotencoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    full_pipeline = ColumnTransformer([
        ("numerical_attribs", numerical_pipeline, numerical_attribs),
        ("categorical_attribs", categorical_pipeline, categorical_attribs)
    ])
    return full_pipeline

if not os.path.exists(MODEL_FILE):
        # WE TRAIN THE DATA
        housing = pd.read_csv("housing.csv")
        housing['income_category'] = pd.cut(housing['median_income'], bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1,2,3,4,5])
        
        split = StratifiedShuffleSplit(n_splits=1, random_state=42, test_size=0.2) 
        
       
        for train_index, test_index in split.split(housing, housing['income_category']):
            
            train_set = housing.loc[train_index]
            test_set = housing.loc[test_index]
            
        
        train_set = train_set.drop('income_category', axis=1)
        
      
        housing_labels = train_set['median_house_value'].copy()
        housing_features = train_set.drop('median_house_value', axis=1)
        
        numerical_attribs = housing_features.drop('ocean_proximity', axis=1).columns.tolist()
        categorical_attribs = ['ocean_proximity']
        
       
        pipeline = build_pipeline(numerical_attribs, categorical_attribs)
        prepared_data = pipeline.fit_transform(housing_features)
        
        model = RandomForestRegressor(random_state=42)
        model.fit(prepared_data, housing_labels)
        
       
        joblib.dump(model, MODEL_FILE)
        joblib.dump(pipeline, PIPELINE_FILE)
        
        print("model trained successfully")
else:
        model = joblib.load(MODEL_FILE)
        pipeline = joblib.load(PIPELINE_FILE)
        input_data = pd.read_csv("input.csv")
        transformed_input = pipeline.transform(input_data)
        prediction = model.predict(transformed_input)
        input_data['median_house_value'] = prediction
        input_data.to_csv("output.csv", index=False)
        print("Inference complete. Results saved to output.csv")
        

Inference complete. Results saved to output.csv
